In [ ]:
# Cell 1: Install dependencies
!pip install -q pymupdf langchain_text_splitters kagglehub tiktoken openai requests


In [ ]:
# Cell 2: Kaggle credentials setup
import shutil, os

src = "./data/kaggle.json"
dest = "/root/.config/kaggle/kaggle.json"
os.makedirs(os.path.dirname(dest), exist_ok=True)

if os.path.exists(src):
    shutil.copy(src, dest)
    os.chmod(dest, 0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = "/root/.config/kaggle"
    print("Kaggle credentials configured.")
else:
    print(f"WARNING: {src} not found. Upload kaggle.json to ./data/ first.")


In [ ]:
# Cell 3: Download Kaggle arxiv dataset (first run ~3GB, cached after)
import kagglehub, glob

dataset_path = kagglehub.dataset_download("Cornell-University/arxiv")
json_files = glob.glob(f"{dataset_path}/**/*.json", recursive=True)
assert json_files, f"No JSON file found in {dataset_path}"
json_path = json_files[0]
print(f"Kaggle dataset ready: {json_path}")


In [ ]:
# Cell 4: Set OpenAI API key (paste your key here)
import os
os.environ["OPENAI_API_KEY"] = "sk-..."   # <-- replace with your key


In [ ]:
# Cell 5: Run the dataset builder
import sys
sys.path.insert(0, ".")   # ensure backend/ is on the path (Colab: set to /content/backend)

from dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(
    output_path="./data/dataset.jsonl",
    chunk_size=700,
    overlap=150,
    samples_per_paper=4,      # ~4 samples per paper
)

builder.build_from_kaggle(
    json_path=json_path,
    n_papers=10,              # change to 200+ for full dataset
    categories=["cs.CL", "cs.AI", "cs.LG", "cs.IR"],
)


In [ ]:
# Cell 6: Inspect results
import json, pandas as pd

samples = [json.loads(l) for l in open("./data/dataset.jsonl", encoding="utf-8")]
print(f"Total samples: {len(samples)}")

df = pd.DataFrame([{
    "paper_id":      s["paper_id"],
    "difficulty":    s["difficulty"],
    "question_type": s["question_type"],
    "section":       s.get("section", ""),
    "answer_len":    len(s["messages"][2]["content"].split()),
} for s in samples])

print("\nDifficulty distribution:")
print(df["difficulty"].value_counts())
print("\nQuestion type distribution:")
print(df["question_type"].value_counts())
df.head(10)
